In [0]:
%run ./transformers/transformMap

In [0]:
from pyspark.sql.functions import *
import delta

In [0]:
class DeltaTableCreator:
    def __init__(self, spark):
        self.spark = spark

    def load(self, source_path, data_format):
        df = (
            spark.read.format(data_format) \
                .load(source_path)
        )

        df_tratado = ( df
          .withColumn("camada_origem", lit("raw"))
          .withColumn("data_ingestao", current_timestamp())
    )

        return df_tratado

    def create_table_with_cdf(
        self,
        df: DataFrame,
        catalog: str,
        schema: str,
        table: str,
        path: str,
        #partition_cols: list = [],
        mode: str = "error"  # or 'overwrite'
    ):
        full_table_name = f"{catalog}.{schema}.{table}"

        # Escreve os dados no caminho delta com CDF ativado
        (
            df.write.format("delta")
            .mode(mode)
            .option("overwriteSchema", "true")
            .option("delta.enableChangeDataFeed", "true")
            .save(path)
        )

        # Cria a tabela no metastore apontando para o caminho
        #partition_stmt = f"PARTITIONED BY ({', '.join(partition_cols)})" if partition_cols else ""
        
        ddl = f"""
        CREATE TABLE IF NOT EXISTS {full_table_name}
        USING DELTA
        LOCATION '{path}'
        TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
        """.strip()

        self.spark.sql(ddl)
        print(f"✅ Tabela {full_table_name} criada com CDF ativado e dados em: {path}")

In [0]:
import delta

class Ingestor:
    def __init__(self, spark, source_path, data_format, target_path, catalog, schemaname, tablename):
        self.spark = spark
        self.source_path = source_path
        self.data_format = data_format
        self.target_path = target_path
        self.catalog = catalog
        self.schemaname = schemaname
        self.tablename = tablename
    
    def load(self, source_path):
        df = (
            spark.read.format(self.data_format) \
                .load(source_path)
        )
        return df

    def saveCDF(self, df):

        df_tratado = ( df
          .withColumn("camada_origem", lit("bronze"))
          .withColumn("data_ingestao", current_timestamp())
    )


        (df_tratado.write
         .format("delta")
         .option("overwriteSchema", "true")
         .mode("overwrite")
         .save(self.target_path)
         )
        return True
    
    def save(self, df):

        df_tratado = ( df
          .withColumn("camada_origem", lit("bronze"))
          .withColumn("data_ingestao", current_timestamp())
    )


        (df_tratado.write
         .format("delta")
         .option("path", self.target_path)
         .option("overwriteSchema", "true")
         .mode("overwrite")
         .saveAsTable(f"{self.catalog}.{self.schemaname}.{self.tablename}")
         )
        return True
    
    def executeLoadAndSave(self, path, flag):
        df = self.load(path)

        return self.save(df) if flag == '0' else self.saveCDF(df)
    
class IncrementalIngestor(Ingestor):
    def __init__(self, spark, source_path, data_format, target_path, catalog, schemaname, tablename, schema_location,checkpoint_location, id_field, timestamp_field):
        super().__init__(spark, source_path, data_format, target_path, catalog, schemaname, tablename)
        self.timestamp_field = timestamp_field
        self.id_field = id_field
        self.checkpoint_location = checkpoint_location
        self.schema_location = schema_location
        self.set_deltatable()
        

    def set_deltatable(self):
        table = f"{self.catalog}.{self.schemaname}.{self.tablename}"
        self.delta_table = delta.DeltaTable.forName(self.spark, table)

    def upsert(self, df):

        df_tratado = ( df
          .withColumn("camada_origem", lit("bronze"))
          .withColumn("data_ingestao", current_timestamp())
        )
        
        df_tratado.createOrReplaceTempView(f"view_{self.tablename}")

        if self.timestamp_field == "N/A":
            query = f"""
        SELECT * 
        from view_{self.tablename}
        """

        else:
            query = f"""
            SELECT * 
            from view_{self.tablename}
            QUALIFY ROW_NUMBER() OVER(PARTITION BY {self.id_field} ORDER BY {self.timestamp_field} DESC) = 1
            """
            
        query_incremental = self.spark.sql(query)

        mergeColumns = [col.strip() for col in self.id_field.split("||")]
        merge_condition = " AND ".join([f"t.{col} = s.{col}" for col in mergeColumns])

        print("executou upsert")

        (
            self.delta_table.alias("t") \
                .merge(
                    query_incremental.alias("s"),
                    merge_condition
                ) \
                .whenMatchedUpdateAll() \
                .whenNotMatchedInsertAll() \
                .execute()
        )

    def load(self, source_path):
       
        df = (
            spark.readStream.format("cloudFiles") \
                .option("cloudFiles.Format", self.data_format) \
                .option('cloudFiles.inferColumnTypes', 'true')
                .option("cloudFiles.schemaLocation", self.schema_location) \
                .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
                .load(source_path)
        )

        print("executou load")

        return df
    
    def save(self, df):
        print("executou save")
        stream = (df.writeStream
         .format("delta")
         .option("checkpointLocation", self.checkpoint_location)
         .option("path", self.target_path)
         .foreachBatch(lambda df, batchId: self.upsert(df))
         .trigger(availableNow=True)
        )
        return stream.start()

    def executeLoadAndSave(self, path):
        print("executou load and save")
        df = self.load(path)
        return self.save(df)

class IngestionCDF:
    def __init__(self, spark, data_format, source_schema, source_table, target_path, catalog, schemaname, tablename,checkpoint_location, id_field, timestamp_field, target_schema):
        #super().__init__(spark, data_format, target_path, catalog, schemaname, tablename)
        self.spark = spark
        self.data_format = data_format
        self.target_path = target_path
        self.catalog = catalog
        self.schemaname = schemaname
        self.source_table = source_table
        self.source_schema = source_schema
        self.target_schema = target_schema
        self.tablename = tablename
        self.timestamp_field = timestamp_field
        self.id_field = id_field
        self.checkpoint_location = checkpoint_location
        self.set_deltatable()
        

    def set_deltatable(self):
        table = f"{self.catalog}.{self.target_schema}.{self.tablename}"
        self.delta_table = delta.DeltaTable.forName(self.spark, table)


    def fullLoadSilver(self, table):

        transformer = TRANSFORM_FUNCTIONS.get(table)

        bronzeSelect = f"""
            select * from {self.catalog}.{self.source_schema}.{self.source_table}
        """
        bronzeInfo = spark.sql(bronzeSelect)

        bronzeTransformed = transformer(bronzeInfo, self.spark) if transformer else bronzeInfo

        (
            bronzeTransformed.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .save(self.target_path) 
        )

        return True


    def load(self, source_path):
       
        df = (
            spark.readStream.format("delta") \
                .option("readChangeFeed", "true") \
                .table(f"{self.catalog}.{self.source_schema}.{self.source_table}")
        )

        print("executou load")

        return df
    
    def save(self, df):
        #print("executou save")
        stream = (df.writeStream
         .option("checkpointLocation", self.checkpoint_location)
         .foreachBatch(lambda df, batchId: self.upsertCDF(df))
         .trigger(availableNow=True)
        )
        return stream.start()
    
    def upsertCDF(self, df):

        df_filtered = df.filter("_change_type <> 'update_preimage'")
        df_filtered.createOrReplaceTempView("df_cdf_filtered")

        # 2. Mantém só a última versão de cada ID por batch
        query_last = f"""
        SELECT *
        FROM df_cdf_filtered
        QUALIFY ROW_NUMBER() OVER (PARTITION BY {self.id_field} ORDER BY _commit_timestamp DESC) = 1
        """
        df_last = self.spark.sql(query_last)
        
        transformer = TRANSFORM_FUNCTIONS.get(self.tablename)
        silverTransformed = transformer(df_last, self.spark) if transformer else df_last

        mergeColumns = [col.strip() for col in self.id_field.split("||")]
        merge_condition = " AND ".join([f"s.{col} = d.{col}" for col in mergeColumns])

        (self.delta_table
             .alias("s")
             .merge(silverTransformed.alias("d"), merge_condition)
             .whenMatchedDelete(condition = "d._change_type = 'delete'")
             .whenMatchedUpdateAll(condition = "d._change_type = 'update_postimage'")
             .whenNotMatchedInsertAll(condition = "d._change_type = 'insert' OR d._change_type = 'update_postimage'")
               .execute())


    def executeLoadAndSaveCDF(self, path):
        print("executou load and save")
        df = self.load(path)
        return self.save(df)


In [0]:
from delta.tables import DeltaTable

class GoldIngestor:
    def __init__(self, spark, source_table, catalog, source_schema, target_schema, gold_table, target_path, checkpoint_path):
        self.spark = spark
        self.source_table = f"{catalog}.{source_schema}.{source_table}"
        self.target_table = f"{catalog}.{target_schema}.{gold_table}"
        self.catalog = catalog
        self.target_schema = target_schema
        self.gold_table = gold_table
        self.target_path = target_path
        self.checkpoint_path = checkpoint_path
        self.set_deltatable()

    def table_exists(self, table_name):
        table_exists = spark.catalog.tableExists(table_name)
        return table_exists

    def transform_with_dims(self, df):
        df.createOrReplaceTempView("silver_cdf")

        transformed = self.spark.sql(f"""
            SELECT 
                p.SK_PORTE_OPERADORA,
                m.SK_MODALIDADE,
                c.SK_COBERTURA,
                t.SK_TRIMESTRE,
                ct.SK_CONTRATACAO,
                ia.SK_ITEM_ASSISTENCIAL,

                s.QT_EVENTOS,
                s.QT_BENEF_FORA_CARENCIA,
                s.VL_DESPESA_ASST_LIQ,
                s.DT_CORTE,
                s.nome_arquivo,
                s.data_modificacao,
                s.data_ingestao,
                s.camada_origem,
                s.data_tratamento,
                s._change_type
            FROM silver_cdf s
            LEFT JOIN {self.catalog}.gold.DIM_PORTE_OPERADORA p ON s.PORTE_OPERADORA = p.NM_PORTE_OPERADORA
            LEFT JOIN {self.catalog}.gold.DIM_MODALIDADE m ON s.GR_MODALIDADE = m.NM_MODALIDADE
            LEFT JOIN {self.catalog}.gold.DIM_COBERTURA c ON s.COBERTURA = c.NM_COBERTURA
            LEFT JOIN {self.catalog}.gold.DIM_TRIMESTRE t ON s.ID_TRIMESTRE = t.ID_TRIMESTRE
            LEFT JOIN {self.catalog}.gold.DIM_CONTRATACAO ct ON s.CONTRATACAO = ct.NM_CONTRATACAO
            LEFT JOIN {self.catalog}.gold.DIM_ITEM_ASSISTENCIAL ia 
                ON s.ID_ITEM_ASST = ia.ID_ITEM_ASST AND s.DE_ITEM_ASST = ia.NM_ITEM_ASSISTENCIAL
        """)

        return transformed

    def set_deltatable(self):
        full_table_name = self.target_table

        if not self.table_exists(full_table_name):
            print(f"⚠️ Tabela {full_table_name} não existe. Criando agora...")

            df_full = (
                self.spark.read
                .format("delta")
                .option("readChangeFeed", "true")
                 .option("startingVersion", 4)
                .table(self.source_table)
                .filter("_change_type <> 'update_preimage'")
            )

            df_transformed = self.transform_with_dims(df_full).drop("_change_type")

            (
                df_transformed.write
                .format("delta")
                .option("overwriteSchema", "true")
                .mode("overwrite")
                .save(self.target_path)
            )

            ddl = f"""
                CREATE TABLE {full_table_name}
                USING DELTA
                LOCATION '{self.target_path}'
            """.strip()
            
            self.spark.sql(ddl)

            print(f"✅ Tabela {full_table_name} criada com sucesso.")

        self.delta_table = DeltaTable.forName(self.spark, full_table_name)

    def load(self):
        return (
            self.spark.readStream
            .format("delta")
            .option("readChangeFeed", "true")
            .table(self.source_table)
            .filter("_change_type <> 'update_preimage'")
        )

    def upsert_gold(self, df):
        df_transformed = self.transform_with_dims(df)
        df_transformed.createOrReplaceTempView("gold_updates")

        query = """
        SELECT *
        FROM gold_updates
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY 
                SK_PORTE_OPERADORA, SK_MODALIDADE, SK_COBERTURA, SK_TRIMESTRE,
                SK_CONTRATACAO, SK_ITEM_ASSISTENCIAL
            ORDER BY _change_type DESC
        ) = 1
        """

        final_df = self.spark.sql(query)

        merge_condition = """
        t.SK_PORTE_OPERADORA = s.SK_PORTE_OPERADORA AND
        t.SK_MODALIDADE = s.SK_MODALIDADE AND
        t.SK_COBERTURA = s.SK_COBERTURA AND
        t.SK_TRIMESTRE = s.SK_TRIMESTRE AND
        t.SK_CONTRATACAO = s.SK_CONTRATACAO AND
        t.SK_ITEM_ASSISTENCIAL = s.SK_ITEM_ASSISTENCIAL
        """

        (
            self.delta_table.alias("t")
            .merge(final_df.alias("s"), merge_condition)
            .whenMatchedDelete(condition="s._change_type = 'delete'")
            .whenMatchedUpdateAll(condition="s._change_type = 'update_postimage'")
            .whenNotMatchedInsertAll(condition="s._change_type IN ('insert', 'update_postimage')")
            .execute()
        )

    def save(self, df):
        return (df.writeStream
            .option("checkpointLocation", self.checkpoint_path)
            .foreachBatch(lambda df, batchId: self.upsert_gold(df))
            .trigger(availableNow=True)
            .start())

    def execute(self):
        df = self.load()
        return self.save(df)
